In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn

data = pd.read_csv('train.csv')

In [2]:
data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
data = data.to_numpy()
# 0th col are the labels of the images
# 1st col up to 784th col are pixel values

In [4]:
data

array([[1, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       ...,
       [7, 0, 0, ..., 0, 0, 0],
       [6, 0, 0, ..., 0, 0, 0],
       [9, 0, 0, ..., 0, 0, 0]], shape=(42000, 785))

In [5]:
labels = data[:, 0]
data = data[:, 1:785]

In [6]:
data.shape

(42000, 784)

In [7]:
train_split = int(0.8 * len(data))
y_train, y_test = labels[:train_split], labels[train_split:]
X_train, X_test = data[:train_split], data[train_split:]

In [8]:
X_train, X_test = torch.tensor(X_train, dtype=torch.float32), torch.tensor(X_test, dtype=torch.float32)
y_train, y_test = torch.tensor(y_train, dtype=torch.int64), torch.tensor(y_test, dtype=torch.int64)
X_train, X_test = X_train/255, X_test/255

In [9]:
class NumClassificationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(784, 10)
        self.layer2 = nn.Linear(10, 10)
        self.layer3 = nn.Linear(10,10)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        x = self.relu(x)
        x = self.layer3(x)
        return x

In [10]:
model_1 = NumClassificationModel()

In [11]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params = model_1.parameters(), lr = 0.1)

In [12]:
X_train.shape

torch.Size([33600, 784])

In [13]:
y_logits = model_1(X_train)
print(y_logits, y_logits.shape)
y_pred = nn.Softmax(dim=1)(y_logits)
print(y_pred, y_pred.shape)
y_pred_labels = torch.argmax(y_pred, dim=1)
print(y_pred_labels, y_pred_labels.shape)

tensor([[ 0.1012, -0.1861,  0.0905,  ...,  0.0700,  0.0030, -0.3115],
        [ 0.1031, -0.2178,  0.0926,  ...,  0.0383, -0.0193, -0.3063],
        [ 0.1002, -0.1888,  0.1066,  ...,  0.0433, -0.0085, -0.3260],
        ...,
        [ 0.1013, -0.2015,  0.0983,  ..., -0.0027, -0.0224, -0.3259],
        [ 0.0969, -0.2024,  0.1015,  ...,  0.0650, -0.0116, -0.3142],
        [ 0.0924, -0.1577,  0.1059,  ...,  0.0035, -0.0072, -0.3501]],
       grad_fn=<AddmmBackward0>) torch.Size([33600, 10])
tensor([[0.1123, 0.0843, 0.1112,  ..., 0.1089, 0.1018, 0.0744],
        [0.1132, 0.0821, 0.1120,  ..., 0.1061, 0.1001, 0.0751],
        [0.1124, 0.0842, 0.1131,  ..., 0.1062, 0.1008, 0.0734],
        ...,
        [0.1127, 0.0833, 0.1124,  ..., 0.1016, 0.0996, 0.0735],
        [0.1123, 0.0833, 0.1129,  ..., 0.1088, 0.1008, 0.0745],
        [0.1112, 0.0866, 0.1127,  ..., 0.1018, 0.1007, 0.0714]],
       grad_fn=<SoftmaxBackward0>) torch.Size([33600, 10])
tensor([6, 6, 6,  ..., 6, 6, 6]) torch.Size([33600])

In [14]:
def accuracy(actual_labels, pred_labels):
    '''
    both actual_labels and pred_labels are vectors where each entry is an example
    '''
    return (pred_labels == actual_labels).sum().item() / len(actual_labels) * 100

In [15]:
# training loop
epochs = 3000

for epoch in range(epochs):
    model_1.train()
    
    # forward pass
    y_logits = model_1(X_train)
    y_pred = nn.Softmax(dim=1)(y_logits)
    y_pred_labels = torch.argmax(y_pred, dim=1)

    # calculate the loss
    loss = loss_fn(y_logits, y_train)
    acc = accuracy(y_train, y_pred_labels)

    # optmizier.zero_grad()
    optimizer.zero_grad()

    # loss.backward()
    loss.backward()

    # optimizer.step()
    optimizer.step()

    # testing loop
    model_1.eval()
    with torch.inference_mode():
        if epoch % 100 == 0: 
            # forward pass
            y_test_logits = model_1(X_test)
            y_test_pred = nn.Softmax(dim=1)(y_test_logits)
            y_test_pred_labels = torch.argmax(y_test_pred, dim=1)

            # calculate the loss
            test_loss = loss_fn(y_test_logits, y_test)
            test_acc = accuracy(y_test, y_test_pred_labels)
            print(f'epoch: {epoch} | train loss: {loss:.5f} | train acc: {acc}% | test loss: {test_loss:.5f} | test acc: {test_acc}%')


epoch: 0 | train loss: 2.32183 | train acc: 9.839285714285714% | test loss: 2.31633 | test acc: 9.892857142857144%
epoch: 100 | train loss: 1.29696 | train acc: 54.830357142857146% | test loss: 1.27701 | test acc: 55.73809523809524%
epoch: 200 | train loss: 0.61630 | train acc: 81.84821428571428% | test loss: 0.60474 | test acc: 82.35714285714286%
epoch: 300 | train loss: 0.48330 | train acc: 86.02678571428571% | test loss: 0.47430 | test acc: 85.98809523809524%
epoch: 400 | train loss: 0.42447 | train acc: 87.76488095238095% | test loss: 0.41662 | test acc: 87.72619047619048%
epoch: 500 | train loss: 0.38917 | train acc: 88.85119047619048% | test loss: 0.38288 | test acc: 88.66666666666667%
epoch: 600 | train loss: 0.36398 | train acc: 89.57738095238095% | test loss: 0.35959 | test acc: 89.32142857142857%
epoch: 700 | train loss: 0.34360 | train acc: 90.11607142857143% | test loss: 0.34101 | test acc: 89.94047619047619%
epoch: 800 | train loss: 0.32590 | train acc: 90.63988095238096% 